# Import Library

In [1]:
import pandas as pd  # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None  # Menonaktifkan peringatan chaining
import numpy as np  # NumPy untuk komputasi numerik
seed = 0
np.random.seed(seed)  # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt  # Matplotlib untuk visualisasi data
import seaborn as sns  # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi

import datetime as dt  # Manipulasi data waktu dan tanggal
import re  # Modul untuk bekerja dengan ekspresi reguler
import string  # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks

!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # Menghapus kata-kata berhenti dalam bahasa Indonesia

from wordcloud import WordCloud  # Membuat visualisasi berbentuk awan kata (word cloud) dari teks
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import logging
from concurrent.futures import ThreadPoolExecutor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.6 MB/s eta 0:00:0000:01


In [2]:
import nltk  # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('punkt_tab')
nltk.download('stopwords')  # Mengunduh dataset yang berisi daftar kata-kata berhenti (stop words) dalam berbagai bahasa.

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load Dataset

In [3]:
# Membuat DataFrame dari hasil scrapreview
youtube_df = pd.read_csv('/kaggle/input/datasets/danielmahulae/data-sentiment/youtube_comments.csv', encoding="utf-8")
youtube_df.shape

(11412, 7)

In [4]:
youtube_df.head()

,video_id,game,comment_id,author_display_name,text,published_at,like_count
0,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzT_UX0Wk_IJF55BHN4AaABAg,@R7Tatsumaki,Untuk nextnya tidak akan banyak Yapping/Typing...,2025-12-08T05:00:08Z,2415
1,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzL-k929aTRVg48OO14AaABAg,@bro_l4na696,Aku tau artinya itu😂😂,2026-02-17T13:48:09Z,0
2,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgyTId8h0XpECen8s3x4AaABAg,@Suggii,"Kebanyakan yapping lu bg, ga satisfying prankNya.",2026-02-17T13:43:53Z,1
3,XAM5nCPwYrI,Mobile Legends: Bang-Bang,Ugxi8Wiraim2xR9cQT14AaABAg,@globalkhaleed,😂😂😂😂😂,2026-02-17T12:17:28Z,0
4,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzUmJdpk34IFotoNJt4AaABAg,@Toji_677,Palir:plr,2026-02-17T06:41:54Z,0


In [5]:
youtube_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11412 entries, 0 to 11411
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   video_id             11412 non-null  object
 1   game                 11412 non-null  object
 2   comment_id           11412 non-null  object
 3   author_display_name  11412 non-null  object
 4   text                 11412 non-null  object
 5   published_at         11412 non-null  object
 6   like_count           11412 non-null  int64 
dtypes: int64(1), object(6)
memory usage: 624.2+ KB


Karena tidak ada baris yang hilang, maka tidak perlu dropna

In [6]:
# Menghapus baris duplikat dari DataFrame clean_df
clean_df = youtube_df.drop_duplicates()

In [7]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11410 entries, 0 to 11411
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   video_id             11410 non-null  object
 1   game                 11410 non-null  object
 2   comment_id           11410 non-null  object
 3   author_display_name  11410 non-null  object
 4   text                 11410 non-null  object
 5   published_at         11410 non-null  object
 6   like_count           11410 non-null  int64 
dtypes: int64(1), object(6)
memory usage: 713.1+ KB


# Preprocessing

In [8]:
def cleaningText(text):

    # fix encoding
    try:
        text = text.encode('latin1').decode('utf-8')
    except:
        pass

    text = text.replace('_', ' ')
    text = re.sub(r'[^\w\s\?\+\!\\ \u2600-\u27BF\U0001f300-\U0001faff]', ' ', text)  
    text = re.sub(r'\b\w*(wk\w*wk|ha\w*ha|kw\w*kw)\w*\b', 'tertawa', text, flags=re.IGNORECASE)

    text = re.sub(r'(.)\1{2,}', r'\1\1', text)# huruf berulang
    text = text.replace('Â²', '2') # ubah pangkat 2 menjadi2
    text = text.replace('²', '2')
    text = re.sub(r'([a-zA-Z]+)2', r'\1 \1', text) # ubah kalimat berulang

    # berikan spase antar tanda baca
    text = text.replace('/', ' atau ')
    text = re.sub(r'([a-zA-Z0-9])([^\w\s])', r'\1 \2', text)
    text = re.sub(r'([^\w\s])([a-zA-Z0-9])', r'\1 \2', text)
    text = re.sub(r'[^\w\s\u2600-\u27BF\U0001f300-\U0001faff](?=[^\s])', ' ', text)
    
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # remove mentions
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # remove hashtag
    text = re.sub(r'RT[\s]', '', text) # remove RT
    text = re.sub(r"http\S+", '', text) # remove link
    
    text = text.replace('\n', ' ') # replace new line into space
    text = re.sub(r'[0-9]+', '', text) # remove numbers
    text = text.strip(' ') # remove characters space from both left and right text
    text = re.sub(r'\s+', ' ', text).strip() # remove spasi ganda
    text = re.sub(r'\b(\w+)(nya|pun)\b', r'\1 \2', text)
    text = re.sub(r'([a-zA-Z])([^\w\s])', r'\1 \2', text)
    text = re.sub(r'([^\w\s])([a-zA-Z])', r'\1 \2', text)

    has_emoji = bool(re.search(r'[\u2600-\u27BF\U0001f300-\U0001faff]', text))
    if len(text) < 3 and not has_emoji:
        return ""
        
    return text

def casefoldingText(text): # Converting all the characters in a text into lower case
    text = text.lower()
    return text

def tokenizingText(text): # Tokenizing or splitting a string, text into a list of tokens
    text = word_tokenize(text)
    return text

def filteringText(text): # Remove stopwords in a text
    listStopwords = set(stopwords.words('indonesian')) # buat ke set daftar stopword indo
    listStopwords1 = set(stopwords.words('english'))   # ambil juga dari inggris
    listStopwords.update(listStopwords1)                # Gabungkan

    manual_stop = {'tuyul','r','p','item','jo','langsung','rayu','kada','ng','tak','fyp','aarrgghh','aa','ta','demon slayer', 'classy',
                   'iya','hai','nang', 'halo','mah','cok','yes','yaa', 'ah','nnya','nya', 'na', 'sih', 'mah','di', 'ya','gasii', 'loh', 'kah', 'woi', 'woii', 'woy'}
    listStopwords.update(manual_stop)

    # menghapus kata negasi dari daftar stopword agar tidak ikut terhapus(untuk pelabelan)
    negasi = {'tidak', 'kurang', 'bukan', 'jangan', 'nggak', 'ga', 'gk', 'no', 'not'}
    for kata in negasi:
        if kata in listStopwords:
            listStopwords.remove(kata)
        
    filtered = [txt for txt in text if (txt not in listStopwords) or (txt.startswith('emoji_'))]
    text = filtered
    return text     # Kembalikan teks yang sudah dibersihkan.

def stemmingText(text): # Reducing a word to its word stem that affixes to suffixes and prefixes or to the roots of words
    # Membuat objek stemmer
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    # daftar kata yang salah stemming di sastrawi
    protected_words = {
        'setuju', 'lanjutin', 'mengetik', 'pakaian','silahkan','silakan','terkenal','dikurangi','kurama','terharu',
        'awalan', 'temuan', 'belajar', 'perubahan', 'beranda','kepotong', 'jawab','menyerah','persia','ternyata'
         }
    
  
    # hapus tanda baca and emot
    text = re.sub(r'[^\w\s]', '', text)
    # Memecah teks menjadi daftar kata
    words = text.split()
    stemmed_words = []
    
    # Menerapkan stemming pada setiap kata dalam daftar
    for word in words:
        if word in protected_words:
            # Jika kata ada di daftar proteksi, ambil aslinya
            stemmed_words.append(word)
        else:
            # Jika tidak, jalankan fungsi stemmer
            stemmed_words.append(stemmer.stem(word))

    # Menggabungkan kata-kata yang telah distem
    stemmed_text = ' '.join(stemmed_words)

    return stemmed_text
    
def toSentence(list_words): # Convert list of words into sentence
    sentence = ' '.join(word for word in list_words)
    return sentence

### Daftar kata-kata slang

In [9]:
slangwords = {"u23": "ayo",'bnag':'abang',"hadehh": "hadeh",'musiikk':'musik',"mass": "abang","yahh": "yah","langsungg": "langsung","hahh": "hah","kerenn": "keren","anjirr": "anjir","tahii": "tahi","bangett": "banget","teriakk": "teriak","guee": "aku","panikk": "panik","liike": "suka","bangg": "abang","takutt": "takut","asemm": "asam","crepyy": "creepy","nyee": "nya","haluss": "kecil","teriaakk": "teriak","anyinkk": "anjir","dongg": "dong","gass": "gas","gilaa": "gila","paraah": "parah","semangatt": "semangat","windahh": "windah","huhh": "huh","batt": "banget","kagett": "kaget", 
              'in game':'dalam permainan',"aslii": "asli", "ngulanfkangmwgwwow": "ngulang kan tertawa","uouuygj h bvv buu": "tertawa","m9m3nt": "momen","b4c0t": "banyak bicara","g4ys": "teman teman","kejangkann": "kejang kan",'titit':'kontol','ush':'perlu','gregett':'greget','kalok':'kalau','jan':'jangan','kjar':'kejar','bundahh':'ibu','gak':'tidak','org':'orang','kethuan':'ketahuan','ngeparank':'ngeprank','dn':'dan','meningal':'meninggal','meninggl':'meninggal','ajg':'anjing','viewers':'viewer', 'rispect':'respect','konl':'kontol','vidclip':'video clip','oulast':'outlast',
              "nike name":"nickname",'nicname':'nickname',"nick name":"nickname","nick nama":"nickname","nik name":"nickname","nikename":"nickname",'bengek':'tertawa','username':'nickname','user name':'nickname','titd':'kontol','trss':'terus','ha nya':'hanya','komuk':'wajah','tolongg':'tolong','gini':'begini','ancrit':'anjir',"bat": "banget",'tiktod':'tiktok','kebayakan':'kebanyakan','typingg':'ngetik','trus':'terus','mekanikk':'mekanik','gmpang':'gampang','curigaii':'curigai','requesh':'request','req':'request','bnyak':'banyak','toxix':'toxic','ig':'instagram','banyan':'banyak','uler':'ular','kripi':'creepy',
              "tumlen":"thumbnail",'g':'tidak','ga':'tidak','peting':'penting','tenang':'santai','kalem':'santai','anjim':'anjir','loncat':'lompat','bocah':'anak kecil','class room':'classroom','kuping':'telinga','bkal':'bakal','bnget':'banget','maz':'abang','orng':'orang','gem':'permainan','piral':'terkenal','horor':'horror','ngapa':'kenapa','geli iii':'geli','kebanyalan' : 'kebanyakan','woyy':'woy','diamon':'diamond','ochimaru':'orochimaru','orchimaru':'orochimaru','orochimaruu':'orochimaru','momochimaru':'orochimaru','irochimaru':'orochimaru','ocorimaru':'orochimaru','orot cihmaru':'orochimaru','cimaru':'orochimaru','orichi maru':'orochimaru','orichimaru':'orochimaru','oci maru':'orochimaru','onochimaru':'orochimaru','orachimaru':'orochimaru','orocimaru':'orochimaru','orocimahi':'orochimaru','diem':'diam','emg':'emang',
              "bhuzet":"buset","r":"R7",'sundaa':'sunda','sidin':'dia','coleb':'collaboration','pleyer':'player','pascoll':'pascol','abg':'abang','bng':'abang','kyak':'seperti','keripi':'creepy','terakan':'teriakan','kliyengan':'pusing','lejend':'legend','lejen':'legend','epik':"epic",'jngn':'jangan','ketaradan':'kelihatan dan','nnton':'menonton','nonton':'menonton','knpe':'kenapa','sm':'sama','jumscare':'jumpscare','prenk':'prank','epictos momentos':'epic momen','kocakk':'kocak','kontl':'kontol','iam':'i am','pke':'pake','mamam':'makan','gbk':'tolol','pu nya':'punya','paenin':'mainkan','moment':'momen',
              "gakuat":"tidak kuat","dolwnod":"download","moster":"monster","exited":"excited","bliek":"be like","bilek":"be like",'gmpng':'gampang','bila':'kalau','mahluk':'makhluk',"mas":"abang","mnding":"mending",'taekk':'tahi','dlm':'dalam','pakek':'pakai','mawi':'romawi','blamde':'blame','bersinarr':'bersinar','p nya':'punya','gimna':'bagaimana','ampe':'sampai','kontrn':'konten','ntr':'nanti','kya':'seperti','epicktoz momentos lezatos':'epic momen','epictod momentod':'epic momen','gimnasium':'gymnasium','gemoyy':'gemoy','kalsel':'kalimantan selatan','kalimantans':'kalimantan',
              "klimantan":"kalimantan",'jadih':'jadi','yt':'youtube','twt':'twitter','bukak':'buka','cepuu':'cepu','kqpan':'kapan','babget':'banget','nton':'menonton','vidio':'video','doong':'dong','deket':'dekat','dideketin':'didekatin','nghonten':'ngonten','gim':'permainan','berutal':'brutal','kids':'anak kecil','kidz':'anak kecil','afk':'away from keyboard','eneg':'muak','maaffg':'maaf','meneonton':'menonton','tik tik ':'tiktok','tik tok':'tiktok','paforit':'favorit','fxxk':'fuck','jugak':'juga','hdir':'hadir','bisasa':'biasa','padaa':'pada','jadii':'jadi',
              "trol":"troll",'ngetrol':'ngetroll','ytb':'youtube','lngsung':'langsung','ngekek':'tertawa','jt':'juta','rb':'ribu','lagii':'lagi','jantungann':'jantungan','njir':'anjir','pen':'pengen','jngan':'jangan','nonto':'menonton','jangenam':'kontol','mab':'mah','rill': 'asli','baangg':'abang','nanyak':'tanya', "na nya": "tanya",'bego':'tolol','begok':'tolol','idah':'sudah','ketahuam':'ketahuan','bamget':'banget','darii':'dari','giama':'bagaimana','solanua':'soal nya','chips':'chip','orgil':'orang gila','dyroth':'dyrroth','dyrot':'dyrroth','dirott':'dyrroth',
              "dirot":"dyrroth",'jilong':'zilong','daksktem':'darksistem','samge':'sange','entar':'nanti','alu':'alucard','alucart':'alucard','proplayer':'pro player','kbnykan':'kebanyakan','gatau':'tidak tahu', 'kurangin':'kurangi','ngegem':'bermain permainan','banh':'abang','denger':'dengar',"apasihh": "apa sih","apasi": "apa sih","apasii": "apa sih",'ngentoott':'ngentot','kentod':'ngentot','ngewri':'ngeri','ngento':'ngentot','ngentoo':'ngentot','nyaa':'nya','teruss':'terus','ingris':'inggris','kknten':'konten','adick':'adik','ngl':'not gonna lie','ti t':'kontol',
              "syerem":"seram",'beytul':'banget','terbru':'terbaru','seruu':'seru','pesbuk':'facebook','ngchat':'ngechat','ajh':'saja','maenin':'mainkan','bateng':'bareng','terurah':'terharu','nii':'ini','gamee':'permainan','artii':'arti','ditiktok':'di tiktok','jirr':'anjir','anjirlah':'anjir lah','gundahh':'gundah','baang':'abang','anjj':'anjing','njj':'anjing','gw':'aku','keget':'kaget','makhlul':'makhluk','ez':'gampang','lopyu':'cinta kamu','caperr':'cari perhatian','plr':'kontol','bangwindahh':'abang windah','plir':'kontol','plir hlus':'kontol kecil','palihalus':'kontol kecil',
              "diaksih":"dikasih",'dikasi':'dikasih','dark system': 'darksistem','dark sistem':'darksistem','ang ang ang':'tertawa','kebanyakanngetik':'kebanyakan ngetik','kethuann':'ketahuan','lhu':'kamu','banyakk':'banyak','ngetikk':'ngetik','soo':'sok','mlu':'melulu','anj':'anjing','eng':'inggris','kebnyakan':'kebanyakan','duluu':'dulu','dh':'dah','dalaam':'dalam', 'kon':'kontol','kristien':'kristen','banjarese':'orang banjar','makasi':'terima kasih','males':'malas','outtlast':'outlast','otlast':'outlast','epictos momentoe':'epic momen','deniel':'denial',
              'geming':'gaming','maghrib':'magrib','nagis':'nangis','ahir':'akhir','akir':'akhir','epig momen':'epic momen','epick momen':'epic momen','abng':'abang','iksn':'ikan', 'gaouka':'tidak suka','nnti':'nanti','ketemi':'ketemu','gada':'tidak ada','gkada':'tidak ada','gud':'good','enddf':'ending','momentos':'momen','epictox momentox':'epic momen','kasian':'kasihan','w':'aku','sol asik':'sok asik',"kliatan": "kelihatan",'bahasa bnjr':'bahasa banjar','titid':'kontol','bner':'bener','ny':'nya','pda':'pada','brisik':'berisik','luu':'kamu','brawel':'brawl',
              'drag sistem':'darksistem','nick':'nickname','kbnyakan':'kebanyakan','ketuan':'ketahuan','njirr':'anjir','dutt':'gendut','ndutt':'gendut','nich nya':'nickname nya','espek':'expect','kbnyakkan':'kebanyakan','ngak':'tidak','cht':'chat','banyaj':'banyak','tetakhir':'terakhir','viewr':'viewer','frank':'prank','masteerr':'master','stremer':'streamer','prang':'prank','pgn':'pengen','ketawan':'ketahuan','kureng':'kurang','geloo':'gila','gelo':'gila','peler':'kontol','ituu':'itu','viwer':'viewer','viwers':'viewer','plaingnga':'paling tidak','wkaka':'tertawa',
              'ke bnyakan cheat':'kebanyakan chat','engga':'tidak','bngt':'banget','smbil':'sambil','benned':'benedetta','bened':'benedetta','benet':'benedetta','benedeta':'benedetta','benetdeta':'benedetta','lahh':'lah','ajaa':'aja','atmin':'admin','herann':'heran',"gje": "tidak jelas",'asikk':'asik','jirrt':'anjir','bkin':'bikin','winda':'windah','bisik':'berisik','bngsat':'bangsat','expreasi':'ekspresi','smngt':'semangat','tok tokku':'tiktok aku','tres':'stres','stress':'stres','awokawoka':'tertawa','jht':'jahat','skrng':'sekarang','mlh':'malah','kuatt':'kuat',
              "sampek": "sampai",'bukannyalainkeran':'bukan nyalain keran','akutakubg':'aku takut abang','sesek':'sesak','gaoaoa':'tidak apa-apa','gapapa':'tidak apa-apa','ga papa':'tidak apa-apa','stetan':'setan','hantu':'setan','bbang':'abang','moyet':'monyet','adyang':'ada yang','subs':'subscribe','trimakasih':'terima kasih','smua':'semua','horol':'horror','teriaak':'teriak','kaloo':'kalau','kelass':'kelas','tkutt':'takut','dikejarr':'dikejar','kmuk':'wajah','vetilasibang':'ventilasi abang','njiir':'anjir',"tumbnailnya":"thumbnail nya",'atu':'atau','cuma':'cuman',
              'keristen':'kristen','widah':'windah','boss':'bos','kbanyakan':'kebanyakan','nm':'nama','dadipada':'daripada','wida':'windah','terseru':'ter seru','tkut':'takut','dud':'gendut','sntai':'santai','ituhh':'itu','serruu':'seru','gmn':'bagaimana','jumpace':'jumpscare','krang':'kurang','konte':'konten','strimer':'streamer','smpe':'sampai','megibur':'menghibur','gaush':'tidak perlu','chatt':'chat','wahh':'wah','bangghh':'abang','ngechatt':'ngechat','jdinyaa':'jadi nya','pecahh':'pecah','balikim':'balikin','ktangkep':'tertangkap','sbenar nya':'sebenar nya',
              'slah': 'salah','blakang':'belakang','sgala':'segala','lebi':'lebih','geme':'permainan','ikann':'ikan','krenn':'keren','smangatt':'semangat','sangking':'saking','kaish':'kasih','winn':'windah','kalao':'kalau','gege':'hebat','kuay':'kuat','terkutut':'terkutuk','anjrr':'anjir','kejangg':'kejang','curig':'curiga','pdhl':'padahal','gils':'gila','skrg':'sekarang','bnr':'benar','anjr':'anjir','nickny':'nickname nya','subscrobe':'subscribe','bcil':'anak kecil','bilin':'bikin','btul':'betul','hbis':'habis','thubnail':'thumbnail','brani':'berani','trea':'teriak',
              'banyk':'banyak','iyaa':'iya','bandell':'bandel','ahkir':'akhir','guwa':'aku','ke bnykan':'kebanyakan','baikk':'baik','luh':'kamu','akuu':'aku','krna':'karena','cokk':'cok','gaes':'teman-teman','gais':'teman-teman','disni':'di sini','dprank':'di prank','viewrs':'viewers','brando':'windah','njer':'anjir','bjir':'anjir','meninggoy':'meninggal','gaseru':'tidak seru','kebangakan':'kebanyakan','mulaa':'mula','awkowaok':'tertawa','heatset':'headset','ggra':'gara-gara','awokawok':'tertawa','ngeprang':'ngeprank','tlol':'tolol','yutuber':'youtuber',
              'gtu':'gitu','komok':'wajah','tauu':'tahu','piwer':'viewer','jangn':'jangan','kbnyk':'kebanyakan','unsup':'batal subscribe','canel':'channel','fb':'facebook','maksih':'makasih','sya':'saya','seruh':'seru','negri':'negeri', 'mukak':'wajah','belakangg':'belakang','seramm':'seram','bngett':'banget','desokripi':'that so creepy','jumpskare':'jumpscare','kuu':'aku','ngent':'ngentot','lahk':'lah','akaun':'akun','nati':'nanti','windh':'windah','muluu':'mulu','dikehar':'dikejar','orochimahi':'orochimaru','sharus':'seharus','otless':'outlast','woee':'woi',
              'oyy':'woi','konteen':'konten','saveg':'savage','segalaa':'segala','wee':'woi','diemm':'diam','ngenntoo t':'ngentot','ngapaa':'kenapa','uintal':'uninstall','bejirr':'anjir','tuhh':'tuh','kebiassan':'kebiasaan','absurr':'absurd','akhirr':'akhir','copott':'lepas','teerpaksa':'terpaksa','sub tittle':'subtitle','anjimm':'anjir','twiter':'twitter','bkp':'bokep','likee':'like','ngitip':'ngintip','seremm':'seram','bangettee':'banget','telingaa':'telinga','persaan':'perasaan'}

slangwords_2 = {"@": "di", "abis": "habis", "wtb": "beli", "masi": "masih", "wts": "jual", "wtt": "tukar", "bgt": "banget", "maks": "maksimal", "plisss": "kumohon", "bgttt": "banget", "indo": "indonesia", "bgtt": "banget", "ad": "ada", "rv": "redvelvet","pllss": "kumohon", "plis": "kumohon", "pls": "kumohon","pliss":"kumohon","plss": "kumohon", "cr": "sumber",'open':'buka', "cod": "bayar ditempat", "adlh": "adalah", "afaik": "as far as i know", "aj": "saja", "ajep-ajep": "dunia gemerlap", "ak": "aku", "akika": "aku", "akkoh": "aku", "akuwh": "aku", "alay": "norak", "alow": "halo", "ambilin": "ambilkan", "ancur": "hancur", "anjrit": "anjir", 'ngumpet': 'sembunyi','ricord':'rekam', 'kang':'tukang',"anter": "antar", 'setek':'panggung',"ap2": "apa-apa", 'setage':'stage',"apasih": "apa sih", "apes": "sial", "aps": "apa", "aq": "aku", "aquwh": "aku", "asbun": "asal bunyi", "aseekk": "asyik", "asekk": "asyik", "asem": "asam", "aspal": "asli tetapi palsu", "astul": "asal tulis", "ato": "atau", "au ah": "tidak mau tahu", "awak": "aku", "ay": "sayang", "ayank": "sayang", "b4": "sebelum", "bakalan": "akan", "bandes": "bantuan desa", "bangedh": "banget", "banpol": "bantuan polisi", "banpur": "bantuan tempur", "basbang": "basi", "bcanda": "bercanda", "bdg": "bandung", "begajulan": "nakal", "beliin": "belikan", 'cing cong': 'cingcong',"bencong": "banci", "bentar": "sebentar", "ber3": "bertiga", "ku":"aku", "beresin": "membereskan", "bete": "bosan", "beud": "banget", "bg": "abang", "bgmn": "bagaimana", "bgt": "banget", "bijimane": "bagaimana", "bintal": "bimbingan mental", "bkl": "akan", "bknnya": "bukan nya", "blegug": "bodoh", "blh": "boleh", "bln": "bulan", "blum": "belum", "bnci": "benci", "bnran": "yang benar", "bodor": "lucu", "bokap": "ayah", "boker": "buang air besar", "bokis": "bohong", "boljug": "boleh juga", "bonek": "bocah nekat", "boyeh": "boleh", "br": "baru", "brg": "bareng", "bro": "saudara laki-laki", "bru": "baru", "bs": "bisa", "bsen": "bosan", "bt": "buat", "btw": "ngomong-ngomong", "buaya": "tidak setia", "bubbu": "tidur", "bubu": "tidur", "bumil": "ibu hamil", "bw": "bawa", "bwt": "buat", "byk": "banyak", "byrin": "bayarkan", "cabal": "sabar", "cadas": "keren", "calo": "makelar", "can": "belum", "capcus": "pergi", "caper": "cari perhatian", "ce": "cewek", "cekal": "cegah tangkal", "cemen": "penakut", "cengengesan": "tertawa", "cepet": "cepat", "cew": "cewek", "chuyunk": "sayang", "cimeng": "ganja", "cipika cipiki": "cium pipi kanan cium pipi kiri", "ciyh": "sih", "ckepp": "cakep", "ckp": "cakep", "cmiiw": "correct me if i'm wrong", "cmpur": "campur", "cong": "banci", "conlok": "cinta lokasi", "cowwyy": "maaf", "cp": "siapa", "cpe": "capek", "cppe": "capek", "cucok": "cocok", "cuex": "cuek", "cumi": "Cuma miscall", "cups": "culun", "curanmor": "pencurian kendaraan bermotor", "curcol": "curahan hati colongan", "cwek": "cewek", "cyin": "cinta", "d": "di", "dah": "deh", "dapet": "dapat", "de": "adik", "dek": "adik", "demen": "suka", "deyh": "deh", "dgn": "dengan", "diancurin": "dihancurkan", "dimaafin": "dimaafkan", "dimintak": "diminta", "disono": "di sana", "dket": "dekat", "dkk": "dan kawan-kawan", "dll": "dan lain-lain", "dlu": "dulu", "dngn": "dengan", "dodol": "bodoh", "doku": "uang", "dongs": "dong", "dpt": "dapat", "dri": "dari", "drmn": "darimana", "drtd": "dari tadi", "dst": "dan seterusnya", "dtg": "datang", "duh": "aduh", "duren": "durian", "ed": "edisi", "egp": "emang gue pikirin", "eke": "aku", "elu": "kamu", "emangnya": "memang nya", "emng": "memang", "endak": "tidak", "enggak": "tidak", "envy": "iri", "ex": "mantan", "fax": "facsimile", "fifo": "first in first out", "folbek": "follow back", "fyi": "sebagai informasi", "gaada": "tidak ada uang", "gag": "tidak", "gaje": "tidak jelas", "gak papa": "tidak apa-apa", "gaptek": "gagap teknologi", 'gprnh':"tidak pernah","gatek": "gagap teknologi", "gawe": "kerja", "gbs": "tidak bisa", "gebetan": "orang yang disuka", "geje": "tidak jelas", "gepeng": "gelandangan dan pengemis", "ghiy": "lagi", "gile": "gila", "gimana": "bagaimana", "gino": "gigi nongol", "githu": "gitu", "gj": "tidak jelas", "gmana": "bagaimana", "gn": "begini", "goblok": "tolol", "golput": "golongan putih", "gowes": "mengayuh sepeda", "gpny": "tidak punya", "gr": "gede rasa", "gretongan": "gratisan", "gtau": "tidak tahu", "gua": "aku", "guoblok": "tolol",  "hallow": "halo", "hankam": "pertahanan dan keamanan", "hehe": "he", "helo": "halo", "hey": "hai", "hlm": "halaman", "hny": "hanya", "hoax": "isu bohong", "hr": "hari", "hrus": "harus", "hubdar": "perhubungan darat", "huff": "mengeluh", "hum": "rumah", "humz": "rumah", "ilang": "hilang", "ilfil": "tidak suka", "imho": "in my humble opinion", "imoetz": "imut", "itungan": "hitungan", "iye": "iya", "ja": "saja", "jadiin": "jadi", "jaim": "jaga image", "jayus": "tidak lucu", "jdi": "jadi", "jem": "jam", "jga": "juga", "jgnkan": "jangankan", "jir": "anjir", "jln": "jalan", "jomblo": "tidak punya pacar", "anjeng":"anjing",'anyink':"anjir", "jubir": "juru bicara", "jutek": "galak", "k": "ke", "kab": "kabupaten", "kabor": "kabur", "kacrut": "kacau", "kadiv": "kepala divisi", "kagak": "tidak", "kalo": "kalau", "kampret": "sialan", "kamtibmas": "keamanan dan ketertiban masyarakat", "kamuwh": "kamu", "kanwil": "kantor wilayah", "karna": "karena", "kasubbag": "kepala subbagian", "katrok": "kampungan", "kayanya": "seperti nya", "kbr": "kabar", "kdu": "harus", "kec": "kecamatan", "kejurnas": "kejuaraan nasional", "kekeuh": "keras kepala", "kel": "kelurahan", "kemaren": "kemarin", "kepengen": "mau", "kepingin": "mau", "kepsek": "kepala sekolah", "kesbang": "kesatuan bangsa", "kesra": "kesejahteraan rakyat", "ketrima": "diterima", "kgiatan": "kegiatan", "kibul": "bohong", "kimpoi": "kawin", "kl": "kalau", "klianz": "kalian", "kloter": "kelompok terbang", "klw": "kalau", "km": "kamu", "kmps": "kampus", "kmrn": "kemarin", "knal": "kenal", "knp": "kenapa", "kodya": "kota madya", "komdis": "komisi disiplin", "komsov": "komunis sovyet", "kongkow": "kumpul bareng teman-teman", "kopdar": "kopi darat", "korup": "korupsi", "kpn": "kapan", "krenz": "keren", "krm": "kirim", "kt": "kita", "ktmu": "ketemu", "ktr": "kantor", "kuper": "kurang pergaulan", "kw": "imitasi","kyk": "seperti", "la": "lah", "lam": "salam", "lamp": "lampiran", "lanud": "landasan udara", "latgab": "latihan gabungan", "leh": "boleh", "lelet": "lambat", "lemot": "lambat", "lgi": "lagi", "lgsg": "langsung", "iat": "lihat","liat": "lihat", "litbang": "penelitian dan pengembangan", "lmyn": "lumayan", "lo": "kamu", "loe": "kamu", "lola": "lambat berfikir", "louph": "cinta", "lp": "lupa", "luber": "langsung, umum, bebas, dan rahasia", "luchuw": "lucu", "lum": "belum", "luthu": "lucu", "lwn": "lawan", "maacih": "terima kasih", "mabal": "bolos", "macem": "macam", "macih": "masih", "maem": "makan", "magabut": "makan gaji buta", "maho": "homo", "mak jang": "kaget", "maksain": "memaksa", "malem": "malam", "mam": "makan", "maneh": "kamu", "maniez": "manis", "mao": "mau", "masukin": "masukkan", "melu": "ikut", "mepet": "dekat sekali", "mgu": "minggu", "migas": "minyak dan gas bumi", "mikol": "minuman beralkohol", "miras": "minuman keras", "mlah": "malah", "mngkn": "mungkin", "mo": "mau", "mokad": "mati", "moso": "masa", "mpe": "sampai", "tungguin": "tunggu", "msk": "masuk","amonali":"anomali","taiik":"tahi","muka":"wajah",'gaakan':'tidak akan',"gede":"besar",'ditmpt':'di tempat','respect':'kagum','bayakan':'banyak','hoaks':'bohong',"mslh": "masalah", "mt": "makan teman", "mubes": "musyawarah besar", "mulu": "melulu", "mumpung": "selagi", "munas": "musyawarah nasional", "muntaber": "muntah dan berak", "musti": "mesti", "muupz": "maaf", "mw": "now watching", "n": "dan", "nanam": "menanam", "nanya": "tanya", "napa": "kenapa", "napi": "narapidana", "napza": "narkotika, alkohol, psikotropika, dan zat adiktif ", "narkoba": "narkotika, psikotropika, dan obat terlarang", "nasgor": "nasi goreng", "nda": "tidak", "ndiri": "sendiri", "ne": "ini", "nekolin": "neokolonialisme", "nembak": "menyatakan cinta", "ngabuburit": "menunggu berbuka puasa", "ngaku": "mengaku", "ngambil": "mengambil", "nganggur": "tidak punya pekerjaan", "ngapah": "kenapa", "ngaret": "terlambat", "ngasih": "memberi", "ngebandel": "berbuat bandel", "ngegosip": "bergosip", "ngeklaim": "mengklaim", "ngeksis": "menjadi eksis", "ngeles": "berkilah", "ngelidur": "menggigau", "ngerampok": "merampok", "ngga": "tidak", "ngibul": "berbohong", "ngiler": "mau", "ngiri": "iri", "ngisiin": "mengisikan", "ngmng": "bicara", "ngomong": "bicara", "ngubek2": "mencari-cari", "ngurus": "mengurus", "nie": "ini", "nih": "ini", "niyh": "nih", "nmr": "nomor", "nntn": "menonton", "nobar": "menonton bareng", "np": "now playing", "ntar": "nanti", "ntn": "menonton", "numpuk": "bertumpuk", "nutupin": "menutupi", "nyari": "mencari", "nyekar": "menyekar", "nyicil": "mencicil", "nyoblos": "mencoblos", "nyokap": "ibu", "ogah": "tidak mau", "ol": "online", "ongkir": "ongkos kirim", "oot": "out of topic", "org2": "orang-orang", "ortu": "orang tua", "otda": "otonomi daerah", "otw": "on the way", "pacal": "pacar", "pake": "pakai","soasik":"sok asyik", "tytyd":"kontol", "pala": "kepala", "pansus": "panitia khusus","sus":"mencurigakan", "parpol": "partai politik", "pasutri": "pasangan suami istri", "pd": "pada", "pede": "percaya diri", "pelatnas": "pemusatan latihan nasional", "pemda": "pemerintah daerah", "pemkot": "pemerintah kota", "pemred": "pemimpin redaksi", "penjas": "pendidikan jasmani", "perda": "peraturan daerah", "perhatiin": "perhatikan", "pesenan": "pesanan", "pgang": "pegang", "pi": "tapi", "pilkada": "pemilihan kepala daerah", "pisan": "sangat", "pk": "penjahat kelamin", "plg": "paling", "pmrnth": "pemerintah", "polantas": "polisi lalu lintas", "ponpes": "pondok pesantren", "pp": "pulang pergi", "prg": "pergi", "prnh": "pernah", "psen": "pesan", "pst": "pasti", "pswt": "pesawat", "pw": "posisi nyaman", "qmu": "kamu", "rakor": "rapat koordinasi", "ranmor": "kendaraan bermotor", "re": "reply", "ref": "referensi", "rehab": "rehabilitasi", "rempong": "sulit", "repp": "balas", "restik": "reserse narkotika", "rhs": "rahasia", "rmh": "rumah", "ru": "baru", "ruko": "rumah toko", "rusunawa": "rumah susun sewa", "ruz": "terus", "saia": "aku", "salting": "salah tingkah", "sampe": "sampai", "samsek": "sama sekali", "sapose": "siapa", "satpam": "satuan pengamanan", "sbb": "sebagai berikut", "sbh": "sebuah", "sbnrny": "sebenar nya", "scr": "secara", "sdgkn": "sedangkan", "sdkt": "sedikit", "se7": "setuju", "sebelas dua belas": "mirip", "sembako": "sembilan bahan pokok", "sempet": "sempat", "sendratari": "seni drama tari", "sgt": "sangat", "shg": "sehingga", "siech": "sih", "sikon": "situasi dan kondisi", "sinetron": "sinema elektronik", "siramin": "siramkan", "sj": "saja", "skalian": "sekalian", "sklh": "sekolah", "skt": "sakit", "slesai": "selesai", "sll": "selalu", "slma": "selama", "slsai": "selesai", "smpt": "sempat", "smw": "semua", "sndiri": "sendiri", "soljum": "sholat jumat", "songong": "sombong", "sory": "maaf", "sosek": "sosial-ekonomi", "sotoy": "sok tahu", "spa": "siapa", "sppa": "siapa", "spt": "seperti", "srtfkt": "sertifikat", "stiap": "setiap", "stlh": "setelah", "suk": "masuk", "sumpek": "sempit", "syg": "sayang", "t4": "tempat", "tajir": "kaya", "tau": "tahu", "taw": "tahu", "td": "tadi", "tdk": "tidak", "teh": "kakak perempuan", "telat": "terlambat", "telmi": "telat berpikir", "temen": "teman", "tengil": "menyebalkan", "tepar": "terkapar", "tggu": "tunggu", "tgu": "tunggu", "thankz": "terima kasih", "thn": "tahun", "tilang": "bukti pelanggaran", "tipiwan": "TvOne", "tks": "terima kasih", "tlp": "telepon", "tls": "tulis", "tmbah": "tambah", "tmen2": "teman-teman", "tmpah": "tumpah", "tmpt": "tempat", "tngu": "tunggu", "tnyta": "ternyata", "tokai": "tai", "toserba": "toko serba ada", "tpi": "tapi", "trdhulu": "terdahulu", "trima": "terima kasih", "trm": "terima", "trs": "terus", "trutama": "terutama", "ts": "penulis", "tst": "tahu sama tahu", "ttg": "tentang", "tuch": "tuh", "tuir": "tua", "tw": "tahu", "u": "kamu", "ud": "sudah", "udah": "sudah", "ujg": "ujung", "ul": "ulangan", "unyu": "lucu", "uplot": "unggah", "urang": "aku", "usah": "perlu", "utk": "untuk", "valas": "valuta asing", "w/": "dengan", "wadir": "wakil direktur", "wamil": "wajib militer", "warkop": "warung kopi", "warteg": "warung tegal", "wat": "buat", "wkt": "waktu", "wtf": "what the fuck", "xixixi": "tertawa", "ya": "iya", "yap": "iya", "yaudah": "ya sudah", "yawdah": "ya sudah", "yg": "yang", "yl": "yang lain", "yo": "iya", 'udh':'sudah',"yowes": "ya sudah", "yup": "iya", "7an": "tujuan", "ababil": "abg labil", "acc": "accord", "adlah": "adalah", "adoh": "aduh", "aha": "tertawa", "aing": "aku", "aja": "saja", "ajj": "saja", "aka": "dikenal juga sebagai", "akko": "aku", "akku": "aku", "akyu": "aku", "aljasa": "asal jadi saja", "ama": "sama", "ambl": "ambil", "ank": "anak", "ap": "apa", "ape": "apa", "aplot": "unggah", "apva": "apa", "aqu": "aku", "asap": "sesegera mungkin", "aseek": "asyik", "asek": "asyik", "aseknya": "asyik nya", "asoy": "asyik", "astrojim": "astagfirullahaladzim", "ath": "kalau begitu", "atuh": "kalau begitu", "ava": "avatar", "aws": "awas", "ayang": "sayang", "ayok": "ayo",'bscot':'banyak bicara', "bacot": "banyak bicara", "bales": "balas", "bangdes": "pembangunan desa", "bangkotan": "tua", "banpres": "bantuan presiden", "bansarkas": "bantuan sarana kesehatan", "bazis": "badan amal, zakat, infak, dan sedekah", "bcoz": "karena", "beb": "sayang", "bejibun": "banyak", "belom": "belum", "bener": "benar", "ber2": "berdua", "berdikari": "berdiri di atas kaki sendiri", "bet": "banget", "beti": "beda tipis", "beut": "banget", "bgd": "banget", "bgs": "bagus", "bhubu": "tidur", "bimbuluh": "bimbingan dan penyuluhan", "bisi": "kalau-kalau", "bkn": "bukan", "bl": "beli", "blg": "bilang", "blm": "belum", "bls": "balas", "bnchi": "benci", "bngung": "bingung", "bnyk": "banyak", "bohay": "badan aduhai", "bokep": "porno", "bokin": "pacar", "bole": "boleh", "bolot": "bodoh", "bonyok": "ayah ibu", "bpk": "bapak", "brb": "segera kembali", "brngkt": "berangkat", "brp": "berapa", "brur": "saudara laki-laki", "bsa": "bisa", "bsk": "besok", "bu_bu": "tidur", "bubarin": "bubarkan", "buber": "buka bersama", "bujubune": "luar biasa", "buser": "buru sergap", "bwhn": "bawahan", "byar": "bayar", "byr": "bayar", "c8": "chat", "cabut": "pergi", "caem": "cakep", "cama-cama": "sama-sama", "cangcut": "celana dalam", "cape": "capek", "caur": "jelek", "cekak": "tidak ada uang", "cekidot": "coba lihat", "cemplungin": "cemplungkan", "ceper": "pendek", "ceu": "kakak perempuan", "cewe": "cewek", "cibuk": "sibuk", "cin": "cinta", "ciye": "cie", "ckck": "ck", "clbk": "cinta lama bersemi kembali", "cmpr": "campur", "cnenk": "senang", "congor": "mulut", "cow": "cowok", "coz": "karena", "cpa": "siapa", "gokil": "gila", "gombal": "suka merayu", "gpl": "tidak pakai lama", "gpp": "tidak apa-apa", "gretong": "gratis", "gt": "begitu", "gtw": "tidak tahu", "gue": "aku", "guys": "teman-teman", "gws": "cepat sembuh", "haghaghag": "tertawa", "hakhak": "tertawa", "handak": "bahan peledak", "hansip": "pertahanan sipil", "hellow": "halo", "helow": "halo", "hi": "hai", "hlng": "hilang", "hnya": "hanya", "houm": "rumah", "hrs": "harus", "hubad": "hubungan angkatan darat", "hubla": "perhubungan laut", "huft": "mengeluh", "humas": "hubungan masyarakat", "idk": "aku tidak tahu", "ilfeel": "tidak suka", "imba": "jago sekali", "imoet": "imut", "info": "informasi", "itung": "hitung", "isengin": "bercanda", "iyala": "iya lah", "iyo": "iya", "jablay": "jarang dibelai", "jadul": "jaman dulu", "jancuk": "anjing", "jd": "jadi", "jdikan": "jadikan", "jg": "juga", "jgn": "jangan", "jijay": "jijik", "jkt": "jakarta", "jnj": "janji", "jth": "jatuh", "jurdil": "jujur adil", "jwb": "jawab", "ka": "kakak", "kabag": "kepala bagian", "kacian": "kasihan", "kadit": "kepala direktorat", "kaga": "tidak", "kaka": "kakak", "kamtib": "keamanan dan ketertiban", "kamuh": "kamu", "kamyu": "kamu", "kapt": "kapten", "kasat": "kepala satuan", "kasubbid": "kepala subbidang", "kau": "kamu", "kbar": "kabar", "kcian": "kasihan", "keburu": "terlanjur", "kedubes": "kedutaan besar", "kek": "seperti", "keknya": "seperti nya", "keliatan": "kelihatan", "keneh": "masih", "kepikiran": "terpikirkan", "kepo": "mau tahu urusan orang", "kere": "tidak punya uang", "kesian": "kasihan", "ketauan": "ketahuan", "keukeuh": "keras kepala", "khan": "kan", "kibus": "kaki busuk", "kk": "kakak", "klian": "kalian", "klo": "kalau", "kluarga": "keluarga", "klwrga": "keluarga", "kmari": "kemari", "kmpus": "kampus", "kn": "kan", "knl": "kenal", "knpa": "kenapa", "kog": "kok", "kompi": "komputer", "komtiong": "komunis Tiongkok", "konjen": "konsulat jenderal", "koq": "kok", "kpd": "kepada", "kptsan": "keputusan", "krik": "garing", "krn": "karena", "ktauan": "ketahuan", "ktny": "kata nya", "kudu": "harus", "kuq": "kok", "ky": "seperti", "kykny": "seperti nya", "laka": "kecelakaan", "lambreta": "lambat", "lansia": "lanjut usia", "lapas": "lembaga pemasyarakatan", "mrinding":"merinding", "lbur": "libur", "lekong": "laki-laki", "lg": "lagi", "lgkp": "lengkap", "lht": "lihat", "linmas": "perlindungan masyarakat", "lmyan": "lumayan", "lngkp": "lengkap",'funny':'lucu', "loch": "loh", "lol": "tolol", "lom": "belum", "loupz": "cinta", "lowh": "kamu", "lu": "kamu", "luchu": "lucu", "luff": "cinta", "luph": "cinta", "lw": "kamu", "lwt": "lewat", "maaciw": "terima kasih", "mabes": "markas besar", "macem-macem": "macam-macam", "madesu": "masa depan suram", "maen": "main", "mahatma": "maju sehat bersama", "mak": "ibu", "makasih": "terima kasih", "malu2in": "memalukan", "mamz": "makan", "manies": "manis", "mantep": "mantap", "markus": "makelar kasus", "mba": "mbak", "mending": "lebih baik", "mgkn": "mungkin", "mhn": "mohon", "miker": "minuman keras", "milis": "mailing list", "mksd": "maksud", "mls": "malas", "mnt": "minta", "moge": "motor gede", "mokat": "mati",'nganggapi':'menanggapi', "mosok": "masa", "msh": "masih", "mskpn": "meskipun", "msng2": "masing-masing", "muahal": "mahal", "muker": "musyawarah kerja", "mumet": "pusing", "muna": "munafik", "munaslub": "musyawarah nasional luar biasa", "musda": "musyawarah daerah", "muup": "maaf", "muuv": "maaf", "nal": "kenal", "nangis": "menangis", "naon": "apa", "napol": "narapidana politik", "naq": "anak", "narsis": "bangga pada diri sendiri", "nax": "anak", "ndak": "tidak", "ndut": "gendut", "nekolim": "neokolonialisme", "nelfon": "menelepon", "ngabis2in": "menghabiskan", "ngambek": "marah", "ngampus": "pergi ke kampus", "ngantri": "mengantri", "ngaruh": "berpengaruh", "ngawur": "berbicara sembarangan", "ngeceng": "kumpul bareng-bareng", "ngeh": "sadar", "ngekos": "tinggal di kos", "ngelamar": "melamar", "ngeliat": "melihat",'pinak':'keturunan', "ngemeng": "bicara terus-terusan", "ngerti": "mengerti", "nggak": "tidak", "ngikut": "ikut", "nginep": "menginap", "ngisi": "mengisi", "ngmg": "bicara", "ngocol": "lucu", "ngomongin": "membicarakan", "ngumpul": "berkumpul", "ni": "ini", "nyasar": "tersesat", "nyariin": "mencari", "nyiapin": "mempersiapkan", "nyiram": "menyiram", "nyok": "ayo", "o/": "oleh", "ok": "ok", "priksa": "periksa", "pro": "profesional", "psn": "pesan", "psti": "pasti", "puanas": "panas", "qmo": "kamu", "qt": "kita", "rame": "ramai", "raskin": "rakyat miskin", "red": "redaksi", "reg": "register", "rejeki": "rezeki", "renstra": "rencana strategis", "reskrim": "reserse kriminal", 'tidakada': 'tidak ada',"sni": "sini", "somse": "sombong sekali", "sorry": "maaf", "sosbud": "sosial-budaya", "sospol": "sosial-politik", "sowry": "maaf", "spd": "sepeda", "sprti": "seperti", "spy": "supaya", "stelah": "setelah", "subbag": "subbagian", "sumbangin": "sumbangkan", "sy": "aku", "syp": "siapa", "tabanas": "tabungan pembangunan nasional", "tar": "nanti", "taun": "tahun", "tawh": "tahu", "tdi": "tadi", "te2p": "tetap", "tekor": "rugi", "telkom": "telekomunikasi", "telp": "telepon", 'pas':'saat',"temen2": "teman-teman", 'yag':'yang',"tengok": "melihat",'sagne':'sange',"terbitin": "terbitkan", "tgl": "tanggal", "thanks": "terima kasih", "thd": "terhadap", "thx": "terima kasih", "tipi": "TV", "tkg": "tukang", "tll": "terlalu", "tlpn": "telepon", "tman": "teman", "tmbh": "tambah", "tmn2": "teman-teman", "tmph": "tumpah", "tnda": "tanda", "tnh": "tanah",'derharu':'terharu', "togel": "toto gelap", "tp": "tapi", "tq": "terima kasih", "trgntg": "tergantung", "trims": "terima kasih", "cb": "coba", "y": "ya", "munfik": "munafik", "reklamuk": "reklamasi", "tren": "trend", "ngehe": "kesal", "mz": "mas", "analisise": "analisis", "sadaar": "sadar", "sept": "september", "nmenarik": "menarik", "zonk": "bodoh", "rights": "benar", "simiskin": "miskin", "ngumpet": "sembunyi", "hardcore": "keras", "akhirx": "akhir nya", "solve": "solusi", "watuk": "batuk", "ngebully": "intimidasi", "masy": "masyarakat", "still": "masih", "tauk": "tahu", "mbual": "bual", "tioghoa": "tionghoa", "kentot": "ngentot", "faktakta": "fakta", "sohib": "teman", "rubahnn": "rubah", "trlalu": "terlalu", "nyela": "cela", "heters": "pembenci", "nyembah": "sembah", "most": "paling", "ikon": "lambang", "light": "terang", "pndukung": "pendukung", "setting": "atur", "seting": "akting", "next": "selanjut nya", "waspadalah": "waspada", "gantengsaya": "ganteng", "parte": "partai", "nyerang": "serang", "nipu": "tipu", "ktipu": "tipu", "jentelmen": "berani", "buangbuang": "buang", "tsangka": "tersangka", "kurng": "kurang", "ista": "nista", "less": "kurang", "koar": "teriak", "paranoid": "takut", "problem": "masalah", "tirani": "tiran", "tilep": "tilap", 'mabar': 'main bersama','noob': 'buruk','gg': 'hebat',"happy": "bahagia", "tak": "tidak", "gk": "tidak", "penertiban": "tertib", "uasai": "kuasa", "mnolak": "tolak", "trending": "trend", "taik": "tahi","taiik": "tahi","Dalaaaaaaaaam": "dalam","yappi": "ngoceh", "yapping": "ngoceh","kda":"skor","palirr":"kontol","palir":"kontol","palirt":"kontol","epep":"free fire", 'mknya':'maka nya',"typing": "ngetik", "tayping": "ngetik","ahokncc": "ahok", "istaa": "nista", "thank" : "terima kasih","thank u" : "terima kasih","benarjujur": "jujur", "anjay": "keren","anjayy": "keren", "satisfying" : "memuaskan", "anying" : "anjir",'hastag':'tagar','vital':'terkenal','vilar':'terkenal','viral':'terkenal','hashtag':'tagar', "kebsnyakan" : "kebanyakan", "jirlah":"anjir lah", "bang":"abang", "p":"halo","tuh" : "itu", "wr" : "win rate", "yaping": "ngoceh", "mgkin": "mungkin", 
"mixing" : "campur", "dut" : "gendut","cupu" : "culun","bocil" : "anak kecil",'mahu':'mau',"otlas":"outlast","outlas":"outlast","bamg" : "abang","mas" : "abang","kak" : "kakak","omong" : "bicara","halus": "kecil","ngprank":"prank","palirhalus" : "kontol kecil", 'meningoy': 'mati',
'tamnel': 'thumbnail','maenin':'mainkan','mainin':'mainkan','cuba':'coba','ikutann':'ikutan','eror':'error','makasii':'terima kasih','salut': 'bangga', 'jagan':'jangan','janagn':'jangan','hati hati':'waspada','bayak':'banyak','moga':"semoga",'ikutann':'ikutan','kayak':'seperti','serem':'seram','sad':'sedih','game':'permainan','terseruu':'seru','tereuuu': 'seru','gedegg': 'kesal', 'ygy': '' ,'yappingg': 'ngoceh',"komen": "komentar","comment": "komentar",
'merehkan':'meremehkan','nga':'tidak','demii':'demi','brothers':'brother','mengaharapakan':'mengharapkan','anu':'itu','ngetikas':'ngetik abang',
'classrooms':'classroom'              }

game_titles = ['brother','fallout','amnesia rebirt','amonaly windah oni','doki doki literature','tiktok','instagram','twitter','facebook',
               'massacre at the mirage','outlast','backroom','zoochosis','bridge of spirits','inside the backrooms','bad piggies bmx boy zombie age',
               'i am bird','classroom','omori','dajjal','horror','emily want to play','pirate the caribbean hunt','kamen rider battride war ',
               'jurassic word','jurassic park', 'pirates of the caribbean', 'pirate', 'dino', 'world','ghaib indonesia','cookie run kingdom',
               'mortal kombat','playroom','allah','tuhan','yesus'
   ]

In [10]:
slang_combined = slangwords | slangwords_2
def fix_slangwords(text):
    sorted_keys = sorted(slang_combined.keys(), key=len, reverse=True)
    
    fixed_text = text
    for slang in sorted_keys:
        # untuk menangani karakter khusus dalam kunci.
        pattern = r'\b' + re.escape(slang) + r'\b'
        fixed_text = re.sub(pattern, slang_combined[slang], fixed_text)
    
    # Merapikan spasi ganda kembali 
    fixed_text = re.sub(r'\s+', ' ', fixed_text).strip()
    return fixed_text


def TitleGame(text):
    for title in game_titles:
        pattern = rf'\b{re.escape(title)}\b'

        def make_title(match):
            return match.group(0).title()
            
        text = re.sub(pattern, make_title, text, flags=re.IGNORECASE)
    
    # 4. Merapikan spasi ganda kembali 
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

Kamus slang atau daftar kata-kata slang ini adalah kumpulan kata-kata slang bahasa Indonesia beserta terjemahan atau artinya dalam bahasa formal. saya juga menambahkan daftar kata-kata slang dari bentuk excel dan digabung dengan slangword yang didefinisikan

In [11]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
clean_df['text_clean'] = clean_df['text'].apply(cleaningText)

# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
clean_df['text_casefoldingText'] = clean_df['text_clean'].apply(casefoldingText)

# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
clean_df['text_slangwords'] = clean_df['text_casefoldingText'].apply(fix_slangwords)

# Mengganti kata-kata yang merupakan sebuah judul Game dengan huruf kapital di awal kata dan menyimpannya di 'text_title'
clean_df['text_title'] = clean_df['text_slangwords'].apply(TitleGame)
clean_df = clean_df[clean_df['text_title'] != ""]

# Mengubah menjadi kata dasar dan menyimpannya di 'text_stemming'
clean_df['text_stemming'] = clean_df['text_title'].apply(stemmingText)

# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
clean_df['text_tokenizingText'] = clean_df['text_stemming'].apply(tokenizingText)

# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
clean_df['text_stopword'] = clean_df['text_tokenizingText'].apply(filteringText)

# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_akhir'
clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)

In [12]:
clean_df

,video_id,game,comment_id,author_display_name,text,published_at,like_count,text_clean,text_casefoldingText,text_slangwords,text_title,text_stemming,text_tokenizingText,text_stopword,text_akhir
0,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzT_UX0Wk_IJF55BHN4AaABAg,@R7Tatsumaki,Untuk nextnya tidak akan banyak Yapping/Typing...,2025-12-08T05:00:08Z,2415,Untuk next nya tidak akan banyak Yapping Typin...,untuk next nya tidak akan banyak yapping typin...,untuk selanjut nya nya tidak akan banyak ngoce...,untuk selanjut nya nya tidak akan banyak ngoce...,untuk lanjut nya nya tidak akan banyak ngoceh ...,"[untuk, lanjut, nya, nya, tidak, akan, banyak,...","[tidak, ngoceh, ngetik, yaak, biar, tidak, col...",tidak ngoceh ngetik yaak biar tidak colok teri...
1,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzL-k929aTRVg48OO14AaABAg,@bro_l4na696,Aku tau artinya itu😂😂,2026-02-17T13:48:09Z,0,Aku tau arti nya itu 😂😂,aku tau arti nya itu 😂😂,aku tahu arti nya itu 😂😂,aku tahu arti nya itu 😂😂,aku tahu arti nya itu,"[aku, tahu, arti, nya, itu]",[arti],arti
2,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgyTId8h0XpECen8s3x4AaABAg,@Suggii,"Kebanyakan yapping lu bg, ga satisfying prankNya.",2026-02-17T13:43:53Z,1,Kebanyakan yapping lu bg ga satisfying prankNya,kebanyakan yapping lu bg ga satisfying pranknya,kebanyakan ngoceh kamu abang tidak memuaskan p...,kebanyakan ngoceh kamu abang tidak memuaskan p...,banyak ngoceh kamu abang tidak muas pranknya,"[banyak, ngoceh, kamu, abang, tidak, muas, pra...","[ngoceh, abang, tidak, muas, pranknya]",ngoceh abang tidak muas pranknya
3,XAM5nCPwYrI,Mobile Legends: Bang-Bang,Ugxi8Wiraim2xR9cQT14AaABAg,@globalkhaleed,😂😂😂😂😂,2026-02-17T12:17:28Z,0,😂😂,😂😂,😂😂,😂😂,,[],[],
4,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzUmJdpk34IFotoNJt4AaABAg,@Toji_677,Palir:plr,2026-02-17T06:41:54Z,0,Palir plr,palir plr,kontol kontol,kontol kontol,kontol kontol,"[kontol, kontol]","[kontol, kontol]",kontol kontol
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11405,d1dH5_zHANY,I Am Fish,UgznQHz4UWVaLGL80354AaABAg,@Glutonnyy,Epic,2021-09-22T11:35:27Z,0,Epic,epic,epic,epic,epic,[epic],[epic],epic
11406,d1dH5_zHANY,I Am Fish,Ugy6ohnlYB_TEmfamHh4AaABAg,@ilhamwindah2174,Blom tamat 🗿,2021-09-22T11:35:26Z,0,Blom tamat 🗿,blom tamat 🗿,blom tamat 🗿,blom tamat 🗿,blom tamat,"[blom, tamat]","[blom, tamat]",blom tamat
11407,d1dH5_zHANY,I Am Fish,UgxNHBZKNi8r43-xd6p4AaABAg,@RizkyAM,Anjayy 🗿🔥,2021-09-22T11:35:24Z,0,Anjayy 🗿🔥,anjayy 🗿🔥,keren 🗿🔥,keren 🗿🔥,keren,[keren],[keren],keren
11409,d1dH5_zHANY,I Am Fish,Ugx18zAtUcVlDKW0L6N4AaABAg,@wawanhermawan556,Aku bang,2021-09-22T11:35:23Z,1,Aku bang,aku bang,aku abang,aku abang,aku abang,"[aku, abang]",[abang],abang


# Pelabelan

In [13]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# Definisikan Mapping Dulu 
id2label = {0: "positive", 1: "neutral", 2: "negative"}
label2id = {"positive": 0, "neutral": 1, "negative": 2}

# Membuat model
model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3,
    id2label=id2label,  
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Load Data
df_train_raw = pd.read_csv("/kaggle/input/datasets/danielmahulae/label-fine-tuning/fine_tune_label.csv", sep=';') 
df_train_raw.columns = df_train_raw.columns.str.lower().str.strip()

# Mapping sentiment teks ke angka menggunakan 
df_train_raw['label'] = df_train_raw['sentiment'].map(label2id)

dataset = Dataset.from_pandas(df_train_raw[['teks_asli', 'label']].rename(columns={'teks_asli': 'text'}))

# Fungsi Tokenisasi
def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_fn, batched=True)
final_dataset = tokenized_dataset # Pakai semua data untuk training

#  Setting Training 
training_args = TrainingArguments(
    output_dir="./hasil_fine_tuning",
    num_train_epochs=5,            
    per_device_train_batch_size=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=10 
)

# Jalankan Training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset
)

trainer.train()

# Simpan
model.save_pretrained("./model_sentimen_V2")
tokenizer.save_pretrained("./model_sentimen_V2")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model_sentimen_V2/tokenizer_config.json',
 './model_sentimen_V2/tokenizer.json')

In [14]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

# Inisialisasi Model V2 
print("Loading Model V2...")
sentiment_model = pipeline(
    "sentiment-analysis", 
    model="./model_sentimen_V2", 
    device=0 if torch.cuda.is_available() else -1
)

# Fungsi Eksekusi Langsung 
def final_process(df):
    df = df.copy()
    
    # Ambil teks langsung dari kolom 'text_title'
    texts = df['text_title'].astype(str).tolist()
    all_labels = []
    all_scores = []

    # Batch processing 
    batch_size = 16
    for i in tqdm(range(0, len(texts), batch_size), desc="Proses Sentiment"):
        batch = texts[i : i + batch_size]
        preds = sentiment_model(batch, truncation=True)
        
        for pred in preds:
            all_labels.append(pred['label'])
            all_scores.append(round(pred['score'], 4))
            
    df['sentiment'] = all_labels
    df['confidence'] = all_scores
    
    return df

# Load raw data
final_df = final_process(clean_df)

# Simpan
final_df.to_csv("Label_komentar_youtube.csv", index=False)

Loading Model V2...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Proses Sentiment: 100%|██████████| 688/688 [01:46<00:00,  6.43it/s]


In [15]:
final_df

,video_id,game,comment_id,author_display_name,text,published_at,like_count,text_clean,text_casefoldingText,text_slangwords,text_title,text_stemming,text_tokenizingText,text_stopword,text_akhir,sentiment,confidence
0,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzT_UX0Wk_IJF55BHN4AaABAg,@R7Tatsumaki,Untuk nextnya tidak akan banyak Yapping/Typing...,2025-12-08T05:00:08Z,2415,Untuk next nya tidak akan banyak Yapping Typin...,untuk next nya tidak akan banyak yapping typin...,untuk selanjut nya nya tidak akan banyak ngoce...,untuk selanjut nya nya tidak akan banyak ngoce...,untuk lanjut nya nya tidak akan banyak ngoceh ...,"[untuk, lanjut, nya, nya, tidak, akan, banyak,...","[tidak, ngoceh, ngetik, yaak, biar, tidak, col...",tidak ngoceh ngetik yaak biar tidak colok teri...,neutral,0.9950
1,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzL-k929aTRVg48OO14AaABAg,@bro_l4na696,Aku tau artinya itu😂😂,2026-02-17T13:48:09Z,0,Aku tau arti nya itu 😂😂,aku tau arti nya itu 😂😂,aku tahu arti nya itu 😂😂,aku tahu arti nya itu 😂😂,aku tahu arti nya itu,"[aku, tahu, arti, nya, itu]",[arti],arti,neutral,0.9884
2,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgyTId8h0XpECen8s3x4AaABAg,@Suggii,"Kebanyakan yapping lu bg, ga satisfying prankNya.",2026-02-17T13:43:53Z,1,Kebanyakan yapping lu bg ga satisfying prankNya,kebanyakan yapping lu bg ga satisfying pranknya,kebanyakan ngoceh kamu abang tidak memuaskan p...,kebanyakan ngoceh kamu abang tidak memuaskan p...,banyak ngoceh kamu abang tidak muas pranknya,"[banyak, ngoceh, kamu, abang, tidak, muas, pra...","[ngoceh, abang, tidak, muas, pranknya]",ngoceh abang tidak muas pranknya,negative,0.9994
3,XAM5nCPwYrI,Mobile Legends: Bang-Bang,Ugxi8Wiraim2xR9cQT14AaABAg,@globalkhaleed,😂😂😂😂😂,2026-02-17T12:17:28Z,0,😂😂,😂😂,😂😂,😂😂,,[],[],,positive,0.9151
4,XAM5nCPwYrI,Mobile Legends: Bang-Bang,UgzUmJdpk34IFotoNJt4AaABAg,@Toji_677,Palir:plr,2026-02-17T06:41:54Z,0,Palir plr,palir plr,kontol kontol,kontol kontol,kontol kontol,"[kontol, kontol]","[kontol, kontol]",kontol kontol,neutral,0.9965
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11405,d1dH5_zHANY,I Am Fish,UgznQHz4UWVaLGL80354AaABAg,@Glutonnyy,Epic,2021-09-22T11:35:27Z,0,Epic,epic,epic,epic,epic,[epic],[epic],epic,neutral,0.9968
11406,d1dH5_zHANY,I Am Fish,Ugy6ohnlYB_TEmfamHh4AaABAg,@ilhamwindah2174,Blom tamat 🗿,2021-09-22T11:35:26Z,0,Blom tamat 🗿,blom tamat 🗿,blom tamat 🗿,blom tamat 🗿,blom tamat,"[blom, tamat]","[blom, tamat]",blom tamat,neutral,0.9963
11407,d1dH5_zHANY,I Am Fish,UgxNHBZKNi8r43-xd6p4AaABAg,@RizkyAM,Anjayy 🗿🔥,2021-09-22T11:35:24Z,0,Anjayy 🗿🔥,anjayy 🗿🔥,keren 🗿🔥,keren 🗿🔥,keren,[keren],[keren],keren,positive,0.9980
11409,d1dH5_zHANY,I Am Fish,Ugx18zAtUcVlDKW0L6N4AaABAg,@wawanhermawan556,Aku bang,2021-09-22T11:35:23Z,1,Aku bang,aku bang,aku abang,aku abang,aku abang,"[aku, abang]",[abang],abang,neutral,0.9972
